In [58]:
import os
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [59]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**TRAIN + EVALUATE POPULARITY MODELS A/B/C**

Inputs :
*   output/popularity_train_dataset.csv
*   output/interaction_train.csv
*   output/interaction_test.cs
Outputs:
*   output/reports/popularity/


Purpose: train Popularity A/B/C and export metrics, charts, report

In [60]:
# Input files from 5-file dataset pipeline
# -------------------------
POPULARITY_TRAIN_FILE = "/content/data/popularity_train_dataset.csv"
INTERACTION_TRAIN_FILE = "/content/data/interaction_train.csv"
INTERACTION_TEST_FILE = "/content/data/interaction_test.csv"

# Optional: if you later have CF/CB/SVD metric CSVs using the same schema,
# put them here and this script will combine them into one comparison report.
EXTERNAL_MODEL_METRIC_FILES = [
    # "output/reports/cf/model_metrics.csv",
    # "output/reports/content_based/model_metrics.csv",
    # "output/reports/svd/model_metrics.csv",
]

REPORT_DIR = "output/reports/popularity"
CHART_DIR = f"{REPORT_DIR}/charts"

In [61]:
# -------------------------
# Model config
# -------------------------
TOP_K_LIST = [5, 10, 20]
PRIMARY_K = 10

W_VIEW = 1.0
W_CART = 3.0
W_PURCHASE = 5.0

DECAY_LAMBDA = 0.01
POPULARITY_WEIGHT = 0.5
BAYESIAN_WEIGHT = 0.5

RELEVANT_RATING_THRESHOLD = 4.0

# If True, a product already seen by the user in train will not be recommended again.
EXCLUDE_TRAIN_SEEN = True

# If your dataset is too large and memory becomes an issue, set this to an int,
# e.g. 10000. Default None evaluates all users in test.
MAX_EVAL_USERS = None

# To avoid huge files, only sample user-level recommendation output.
SAVE_RECOMMENDATION_SAMPLE = True
RECOMMENDATION_SAMPLE_USERS = 100

# Save per-user metrics for analysis. For very large user counts, set False.
SAVE_USER_METRICS = True

CHUNKSIZE = 100_000

In [62]:
# ============================================================
# BASIC UTILITIES
# ============================================================

def ensure_input_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Input file not found: {path}")


def ensure_dirs():
    os.makedirs(REPORT_DIR, exist_ok=True)
    os.makedirs(CHART_DIR, exist_ok=True)


def safe_minmax(series):
    s = pd.to_numeric(series, errors="coerce").fillna(0)
    min_v = s.min()
    max_v = s.max()
    if pd.isna(min_v) or pd.isna(max_v) or max_v == min_v:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - min_v) / (max_v - min_v)


def read_csv_required(path, required_cols):
    ensure_input_file(path)
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")
    return df

In [63]:
# ============================================================
# LOAD POPULARITY TRAIN DATA
# ============================================================

def load_popularity_train_dataset():
    required_cols = [
        "ProductId",
        "ViewCount",
        "CartCount",
        "PurchaseCount",
        "AvgRating",
        "RatingCount",
        "LastTimestamp",
        "LastDate",
    ]

    df = read_csv_required(POPULARITY_TRAIN_FILE, required_cols)

    numeric_cols = [
        "ViewCount",
        "CartCount",
        "PurchaseCount",
        "AvgRating",
        "RatingCount",
        "LastTimestamp",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["LastDate"] = pd.to_datetime(df["LastDate"], errors="coerce")

    df = df.dropna(
        subset=[
            "ProductId",
            "ViewCount",
            "CartCount",
            "PurchaseCount",
            "AvgRating",
            "RatingCount",
            "LastTimestamp",
            "LastDate",
        ]
    ).copy()

    df = df[df["RatingCount"] > 0].copy()
    df["ProductId"] = df["ProductId"].astype(str).str.strip()
    df = df[df["ProductId"].str.len() > 0].copy()

    print(f"Loaded popularity train products: {len(df):,}", flush=True)
    return df

In [64]:
# ================
# TRAIN MODELS A
# ================
def train_model_a_popularity_only(product_df):
    """
    MODEL A: POPULARITY ONLY
    Score_A = View + 3*Cart + 5*Purchase
    """
    df = product_df.copy()

    df["Score_A"] = (
        W_VIEW * df["ViewCount"]
        + W_CART * df["CartCount"]
        + W_PURCHASE * df["PurchaseCount"]
    )

    df["PredRating_A"] = df["AvgRating"]

    df = df.sort_values("Score_A", ascending=False).reset_index(drop=True)
    df["Rank_A"] = df.index + 1

    return df

In [65]:
# ================
# TRAIN MODELS B
# ================
def train_model_b_popularity_time_decay(product_df):
    """
    MODEL B: POPULARITY + TIME DECAY
    Score_B = PopularityScore * DecayFactor
    """
    df = product_df.copy()

    latest_date = df["LastDate"].max()
    df["AgeInDays"] = (latest_date - df["LastDate"]).dt.days.clip(lower=0)

    df["PopularityScore"] = (
        W_VIEW * df["ViewCount"]
        + W_CART * df["CartCount"]
        + W_PURCHASE * df["PurchaseCount"]
    )

    df["DecayFactor"] = np.exp(-DECAY_LAMBDA * df["AgeInDays"])
    df["Score_B"] = df["PopularityScore"] * df["DecayFactor"]
    df["PredRating_B"] = df["AvgRating"]

    df = df.sort_values("Score_B", ascending=False).reset_index(drop=True)
    df["Rank_B"] = df.index + 1

    return df

In [66]:
# ================
# TRAIN MODELS C
# ================
def train_model_c_rank_based_bayesian(product_df):
    """
    MODEL C: RANK-BASED POPULARITY + BAYESIAN AVERAGE
    Score_C = 0.5*PopularityDecayNorm + 0.5*BayesianNorm
    """
    df = product_df.copy()

    latest_date = df["LastDate"].max()
    df["AgeInDays"] = (latest_date - df["LastDate"]).dt.days.clip(lower=0)

    df["PopularityScore"] = (
        W_VIEW * df["ViewCount"]
        + W_CART * df["CartCount"]
        + W_PURCHASE * df["PurchaseCount"]
    )

    df["DecayFactor"] = np.exp(-DECAY_LAMBDA * df["AgeInDays"])
    df["PopularityDecayScore"] = df["PopularityScore"] * df["DecayFactor"]

    global_avg_rating = df["AvgRating"].mean()
    k = df["RatingCount"].mean()

    df["BayesianAvg"] = (
        global_avg_rating * k
        + df["AvgRating"] * df["RatingCount"]
    ) / (k + df["RatingCount"])

    df["PopularityDecayNorm"] = safe_minmax(df["PopularityDecayScore"])
    df["BayesianNorm"] = safe_minmax(df["BayesianAvg"])

    df["Score_C"] = (
        POPULARITY_WEIGHT * df["PopularityDecayNorm"]
        + BAYESIAN_WEIGHT * df["BayesianNorm"]
    )

    df["PredRating_C"] = df["BayesianAvg"]

    df = df.sort_values("Score_C", ascending=False).reset_index(drop=True)
    df["Rank_C"] = df.index + 1

    return df

In [67]:
# ============================================================
# PREPARE GLOBAL RANKED LISTS
# ============================================================

def build_ranked_lists(model_a_df, model_b_df, model_c_df):
    ranked_lists = {
        "Popularity_A_Only": model_a_df["ProductId"].tolist(),
        "Popularity_B_TimeDecay": model_b_df["ProductId"].tolist(),
        "Popularity_C_Bayesian": model_c_df["ProductId"].tolist(),
    }
    return ranked_lists

In [68]:
# ============================================================
# LOAD TRAIN SEEN + TEST ACTUALS
# ============================================================

def load_train_seen_items():
    """
    Build user -> set(ProductId) from interaction_train.csv.
    Used to avoid recommending items already seen in train.
    """
    if not EXCLUDE_TRAIN_SEEN:
        return {}

    ensure_input_file(INTERACTION_TRAIN_FILE)
    seen = defaultdict(set)
    total_rows = 0

    usecols = ["UserId", "ProductId"]

    for i, chunk in enumerate(pd.read_csv(INTERACTION_TRAIN_FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False)):
        chunk["UserId"] = chunk["UserId"].astype(str).str.strip()
        chunk["ProductId"] = chunk["ProductId"].astype(str).str.strip()
        chunk = chunk.dropna(subset=["UserId", "ProductId"])

        for user_id, product_id in zip(chunk["UserId"], chunk["ProductId"]):
            if user_id and product_id:
                seen[user_id].add(product_id)

        total_rows += len(chunk)
        print(f"train seen chunk {i + 1}: processed {total_rows:,} rows", flush=True)

    print(f"Loaded train seen users: {len(seen):,}", flush=True)
    return seen


def load_test_actual_items():
    """
    Build user -> set(relevant ProductId) from interaction_test.csv.
    Relevant item = IsRelevant == 1 if the column exists, otherwise Rating >= threshold.
    Also return all test users.
    """
    ensure_input_file(INTERACTION_TEST_FILE)

    actual = defaultdict(set)
    test_users = set()
    total_rows = 0
    relevant_rows = 0

    for i, chunk in enumerate(pd.read_csv(INTERACTION_TEST_FILE, chunksize=CHUNKSIZE, low_memory=False)):
        chunk.columns = chunk.columns.str.strip()

        required_cols = ["UserId", "ProductId", "Rating"]
        missing = [c for c in required_cols if c not in chunk.columns]
        if missing:
            raise ValueError(f"{INTERACTION_TEST_FILE} is missing columns: {missing}")

        chunk["UserId"] = chunk["UserId"].astype(str).str.strip()
        chunk["ProductId"] = chunk["ProductId"].astype(str).str.strip()
        chunk["Rating"] = pd.to_numeric(chunk["Rating"], errors="coerce")
        chunk = chunk.dropna(subset=["UserId", "ProductId", "Rating"])

        if "IsRelevant" in chunk.columns:
            chunk["IsRelevant"] = pd.to_numeric(chunk["IsRelevant"], errors="coerce").fillna(0).astype(int)
        else:
            chunk["IsRelevant"] = (chunk["Rating"] >= RELEVANT_RATING_THRESHOLD).astype(int)

        for user_id in chunk["UserId"].unique():
            test_users.add(user_id)

        rel_chunk = chunk[chunk["IsRelevant"] == 1]
        for user_id, product_id in zip(rel_chunk["UserId"], rel_chunk["ProductId"]):
            if user_id and product_id:
                actual[user_id].add(product_id)

        total_rows += len(chunk)
        relevant_rows += len(rel_chunk)
        print(
            f"test actual chunk {i + 1}: rows {total_rows:,} | relevant {relevant_rows:,}",
            flush=True,
        )

    # Evaluate only users with at least one relevant item in test.
    actual = {u: items for u, items in actual.items() if len(items) > 0}

    if MAX_EVAL_USERS is not None:
        selected_users = list(actual.keys())[:MAX_EVAL_USERS]
        actual = {u: actual[u] for u in selected_users}

    print(f"Loaded test users with relevant items: {len(actual):,}", flush=True)
    return actual, test_users

In [69]:
# ============================================================
# RECOMMENDATION + METRICS
# ============================================================

def recommend_for_user(global_ranked_products, seen_items, k):
    recs = []
    seen_items = seen_items or set()

    for product_id in global_ranked_products:
        if EXCLUDE_TRAIN_SEEN and product_id in seen_items:
            continue
        recs.append(product_id)
        if len(recs) >= k:
            break

    return recs


def precision_at_k(recs, actual, k):
    if k == 0:
        return 0.0
    return len(set(recs[:k]) & actual) / k


def recall_at_k(recs, actual, k):
    if len(actual) == 0:
        return 0.0
    return len(set(recs[:k]) & actual) / len(actual)


def f1_at_k(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def hit_rate_at_k(recs, actual, k):
    return 1.0 if len(set(recs[:k]) & actual) > 0 else 0.0


def average_precision_at_k(recs, actual, k):
    if len(actual) == 0:
        return 0.0

    score = 0.0
    hits = 0

    for idx, product_id in enumerate(recs[:k], start=1):
        if product_id in actual:
            hits += 1
            score += hits / idx

    if hits == 0:
        return 0.0

    return score / min(len(actual), k)


def reciprocal_rank_at_k(recs, actual, k):
    for idx, product_id in enumerate(recs[:k], start=1):
        if product_id in actual:
            return 1.0 / idx
    return 0.0


def ndcg_at_k(recs, actual, k):
    dcg = 0.0
    for idx, product_id in enumerate(recs[:k], start=1):
        rel = 1.0 if product_id in actual else 0.0
        if rel > 0:
            dcg += rel / math.log2(idx + 1)

    ideal_hits = min(len(actual), k)
    if ideal_hits == 0:
        return 0.0

    idcg = sum(1.0 / math.log2(idx + 1) for idx in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_ranking_model(model_name, ranked_products, actual_by_user, train_seen_by_user, k_list):
    rows = []
    user_metric_rows = []
    sample_recommendations = []

    max_k = max(k_list)
    users = list(actual_by_user.keys())

    for user_index, user_id in enumerate(users, start=1):
        actual = actual_by_user[user_id]
        seen = train_seen_by_user.get(user_id, set()) if EXCLUDE_TRAIN_SEEN else set()
        recs = recommend_for_user(ranked_products, seen, max_k)

        if SAVE_RECOMMENDATION_SAMPLE and user_index <= RECOMMENDATION_SAMPLE_USERS:
            sample_recommendations.append({
                "Model": model_name,
                "UserId": user_id,
                "RecommendedProducts": "|".join(recs),
                "ActualRelevantProducts": "|".join(list(actual)[:max_k]),
            })

        for k in k_list:
            p = precision_at_k(recs, actual, k)
            r = recall_at_k(recs, actual, k)
            f1 = f1_at_k(p, r)
            hr = hit_rate_at_k(recs, actual, k)
            ap = average_precision_at_k(recs, actual, k)
            rr = reciprocal_rank_at_k(recs, actual, k)
            ndcg = ndcg_at_k(recs, actual, k)

            user_metric_rows.append({
                "Model": model_name,
                "UserId": user_id,
                "K": k,
                "Precision@K": p,
                "Recall@K": r,
                "F1@K": f1,
                "HitRate@K": hr,
                "AP@K": ap,
                "RR@K": rr,
                "NDCG@K": ndcg,
                "ActualRelevantCount": len(actual),
            })

        if user_index % 5000 == 0:
            print(f"{model_name}: evaluated {user_index:,}/{len(users):,} users", flush=True)

    user_metrics_df = pd.DataFrame(user_metric_rows)

    for k in k_list:
        k_df = user_metrics_df[user_metrics_df["K"] == k]
        recommended_catalog = set(ranked_products[:k])

        rows.append({
            "Model": model_name,
            "K": k,
            "EvaluatedUsers": len(users),
            "Precision@K": k_df["Precision@K"].mean(),
            "Recall@K": k_df["Recall@K"].mean(),
            "F1@K": k_df["F1@K"].mean(),
            "HitRate@K": k_df["HitRate@K"].mean(),
            "MAP@K": k_df["AP@K"].mean(),
            "MRR@K": k_df["RR@K"].mean(),
            "NDCG@K": k_df["NDCG@K"].mean(),
            "CatalogCoverage@K": len(recommended_catalog) / len(ranked_products) if len(ranked_products) > 0 else 0,
        })

    summary_df = pd.DataFrame(rows)
    sample_df = pd.DataFrame(sample_recommendations)

    return summary_df, user_metrics_df, sample_df


In [70]:
# ============================================================
# RATING PREDICTION METRICS: RMSE / MAE
# ============================================================

def build_prediction_maps(model_a_df, model_b_df, model_c_df):
    return {
        "Popularity_A_Only": model_a_df.set_index("ProductId")["PredRating_A"].to_dict(),
        "Popularity_B_TimeDecay": model_b_df.set_index("ProductId")["PredRating_B"].to_dict(),
        "Popularity_C_Bayesian": model_c_df.set_index("ProductId")["PredRating_C"].to_dict(),
    }


def evaluate_rating_prediction(prediction_maps):
    ensure_input_file(INTERACTION_TEST_FILE)

    stats = {
        model_name: {
            "count": 0,
            "squared_error_sum": 0.0,
            "absolute_error_sum": 0.0,
        }
        for model_name in prediction_maps.keys()
    }

    for i, chunk in enumerate(pd.read_csv(INTERACTION_TEST_FILE, chunksize=CHUNKSIZE, low_memory=False)):
        chunk.columns = chunk.columns.str.strip()
        chunk["ProductId"] = chunk["ProductId"].astype(str).str.strip()
        chunk["Rating"] = pd.to_numeric(chunk["Rating"], errors="coerce")
        chunk = chunk.dropna(subset=["ProductId", "Rating"])

        for model_name, pred_map in prediction_maps.items():
            preds = chunk["ProductId"].map(pred_map)
            valid_mask = preds.notna()

            if valid_mask.sum() == 0:
                continue

            y_true = chunk.loc[valid_mask, "Rating"].astype(float)
            y_pred = preds.loc[valid_mask].astype(float)

            errors = y_true - y_pred
            stats[model_name]["count"] += len(errors)
            stats[model_name]["squared_error_sum"] += float(np.sum(errors ** 2))
            stats[model_name]["absolute_error_sum"] += float(np.sum(np.abs(errors)))

        print(f"rating prediction chunk {i + 1}: processed {len(chunk):,} rows", flush=True)

    rows = []
    for model_name, s in stats.items():
        if s["count"] == 0:
            rmse = np.nan
            mae = np.nan
        else:
            rmse = math.sqrt(s["squared_error_sum"] / s["count"])
            mae = s["absolute_error_sum"] / s["count"]

        rows.append({
            "Model": model_name,
            "RMSE": rmse,
            "MAE": mae,
            "RatingEvalRows": s["count"],
        })

    return pd.DataFrame(rows)

In [71]:

# ============================================================
# TOP PRODUCTS + RANK COMPARISON
# ============================================================

def build_rank_comparison(model_a_df, model_b_df, model_c_df):
    base_cols = ["ProductId"]
    optional_cols = ["Title", "MainCategory", "SourceCategory", "AvgRating", "RatingCount"]

    rank_a = model_a_df[base_cols + ["Rank_A", "Score_A"]].copy()
    rank_b = model_b_df[base_cols + ["Rank_B", "Score_B"]].copy()

    c_cols = base_cols + [
        "Rank_C",
        "Score_C",
        "BayesianAvg",
        "PopularityDecayScore",
        "PopularityDecayNorm",
        "BayesianNorm",
    ]

    for col in optional_cols:
        if col in model_c_df.columns:
            c_cols.append(col)

    rank_c = model_c_df[c_cols].copy()

    compare = (
        rank_a
        .merge(rank_b, on="ProductId", how="inner")
        .merge(rank_c, on="ProductId", how="inner")
    )

    compare["Change_A_to_C"] = compare["Rank_A"] - compare["Rank_C"]
    compare["Change_B_to_C"] = compare["Rank_B"] - compare["Rank_C"]

    compare = compare.sort_values("Rank_C", ascending=True).reset_index(drop=True)
    return compare


def build_top_products(model_df, model_name, score_col, rank_col, top_n=20):
    cols = ["ProductId", rank_col, score_col, "AvgRating", "RatingCount"]
    for col in ["Title", "MainCategory", "SourceCategory", "Price"]:
        if col in model_df.columns:
            cols.append(col)

    out = model_df.head(top_n)[cols].copy()
    out.insert(0, "Model", model_name)
    return out

In [72]:
# ============================================================
# CHARTS
# ============================================================

SAVE_GROUPED_DASHBOARD = True
SAVE_SEPARATE_CHARTS = False


def save_grouped_ranking_metrics_chart(metrics_df, k=PRIMARY_K):
    """
    One grouped chart for ranking metrics:
    Precision, Recall, F1, HitRate, MAP, MRR, NDCG.
    """

    metrics = [
        "Precision@K",
        "Recall@K",
        "F1@K",
        "HitRate@K",
        "MAP@K",
        "MRR@K",
        "NDCG@K"
    ]

    df = metrics_df[metrics_df["K"] == k].copy()

    if len(df) == 0:
        print("No data for grouped ranking metrics chart.")
        return

    x = np.arange(len(metrics))
    width = 0.25

    plt.figure(figsize=(16, 8))

    for idx, (_, row) in enumerate(df.iterrows()):
        values = [row[m] for m in metrics]
        plt.bar(
            x + (idx - 1) * width,
            values,
            width,
            label=row["Model"]
        )

    plt.xticks(x, metrics, rotation=30, ha="right")
    plt.ylabel("Score")
    plt.title(f"Grouped Ranking Metrics Comparison @ {k}")
    plt.legend()
    plt.tight_layout()

    path = f"{CHART_DIR}/grouped_ranking_metrics_k{k}.png"
    plt.savefig(path, dpi=150)
    plt.close()

    print("Saved chart:", path, flush=True)


def save_grouped_error_chart(rating_df):
    """
    One grouped chart for RMSE and MAE.
    """

    if len(rating_df) == 0:
        print("No data for grouped error chart.")
        return

    x = np.arange(len(rating_df))
    width = 0.35

    plt.figure(figsize=(12, 6))

    plt.bar(
        x - width / 2,
        rating_df["RMSE"],
        width,
        label="RMSE"
    )

    plt.bar(
        x + width / 2,
        rating_df["MAE"],
        width,
        label="MAE"
    )

    plt.xticks(
        x,
        rating_df["Model"],
        rotation=20,
        ha="right"
    )

    plt.ylabel("Error")
    plt.title("Grouped Rating Prediction Error Comparison")
    plt.legend()
    plt.tight_layout()

    path = f"{CHART_DIR}/grouped_error_metrics.png"
    plt.savefig(path, dpi=150)
    plt.close()

    print("Saved chart:", path, flush=True)


def save_grouped_top_rank_score_chart(
    model_a_df,
    model_b_df,
    model_c_df,
    top_n=10
):
    """
    Compare normalized top-rank scores of A/B/C by rank position.
    This avoids the problem that Score_A, Score_B, Score_C have different scales.
    """

    a = model_a_df.head(top_n)[["Rank_A", "Score_A"]].copy()
    b = model_b_df.head(top_n)[["Rank_B", "Score_B"]].copy()
    c = model_c_df.head(top_n)[["Rank_C", "Score_C"]].copy()

    a["Rank"] = range(1, len(a) + 1)
    b["Rank"] = range(1, len(b) + 1)
    c["Rank"] = range(1, len(c) + 1)

    a["NormalizedScore"] = safe_minmax(a["Score_A"])
    b["NormalizedScore"] = safe_minmax(b["Score_B"])
    c["NormalizedScore"] = safe_minmax(c["Score_C"])

    plt.figure(figsize=(12, 6))

    plt.plot(
        a["Rank"],
        a["NormalizedScore"],
        marker="o",
        label="Model A"
    )

    plt.plot(
        b["Rank"],
        b["NormalizedScore"],
        marker="o",
        label="Model B"
    )

    plt.plot(
        c["Rank"],
        c["NormalizedScore"],
        marker="o",
        label="Model C"
    )

    plt.xlabel("Rank position")
    plt.ylabel("Normalized score")
    plt.title(f"Top-{top_n} Normalized Score Trend by Rank")
    plt.xticks(range(1, top_n + 1))
    plt.legend()
    plt.tight_layout()

    path = f"{CHART_DIR}/grouped_top_rank_scores.png"
    plt.savefig(path, dpi=150)
    plt.close()

    print("Saved chart:", path, flush=True)


def save_grouped_topk_overlap_chart(
    model_a_df,
    model_b_df,
    model_c_df,
    k=PRIMARY_K
):
    """
    One grouped chart showing overlap rate between model top-K lists.
    """

    top_a = set(model_a_df.head(k)["ProductId"])
    top_b = set(model_b_df.head(k)["ProductId"])
    top_c = set(model_c_df.head(k)["ProductId"])

    labels = [
        "A vs B",
        "A vs C",
        "B vs C"
    ]

    values = [
        len(top_a & top_b) / k,
        len(top_a & top_c) / k,
        len(top_b & top_c) / k
    ]

    plt.figure(figsize=(8, 5))
    plt.bar(labels, values)
    plt.ylim(0, 1)
    plt.ylabel("Overlap rate")
    plt.title(f"Top-{k} Overlap Rate Between Models")
    plt.tight_layout()

    path = f"{CHART_DIR}/grouped_topk_overlap_k{k}.png"
    plt.savefig(path, dpi=150)
    plt.close()

    print("Saved chart:", path, flush=True)


def save_dashboard_chart(
    metrics_df,
    rating_df,
    model_a_df,
    model_b_df,
    model_c_df,
    k=PRIMARY_K
):
    """
    One dashboard image containing:
    1. Ranking metrics
    2. RMSE/MAE
    3. Top-K overlap
    4. Top-rank normalized score trend
    """

    metrics = [
        "Precision@K",
        "Recall@K",
        "F1@K",
        "HitRate@K",
        "MAP@K",
        "MRR@K",
        "NDCG@K"
    ]

    ranking_df = metrics_df[metrics_df["K"] == k].copy()

    fig, axes = plt.subplots(2, 2, figsize=(20, 12))

    # -------------------------
    # Chart 1: Ranking metrics
    # -------------------------

    ax1 = axes[0, 0]

    x = np.arange(len(metrics))
    width = 0.25

    for idx, (_, row) in enumerate(ranking_df.iterrows()):
        values = [row[m] for m in metrics]
        ax1.bar(
            x + (idx - 1) * width,
            values,
            width,
            label=row["Model"]
        )

    ax1.set_title(f"Ranking Metrics @ {k}")
    ax1.set_ylabel("Score")
    ax1.set_xticks(x)
    ax1.set_xticklabels(metrics, rotation=30, ha="right")
    ax1.legend()

    # -------------------------
    # Chart 2: RMSE / MAE
    # -------------------------

    ax2 = axes[0, 1]

    error_x = np.arange(len(rating_df))
    error_width = 0.35

    ax2.bar(
        error_x - error_width / 2,
        rating_df["RMSE"],
        error_width,
        label="RMSE"
    )

    ax2.bar(
        error_x + error_width / 2,
        rating_df["MAE"],
        error_width,
        label="MAE"
    )

    ax2.set_title("Rating Prediction Error")
    ax2.set_ylabel("Error")
    ax2.set_xticks(error_x)
    ax2.set_xticklabels(
        rating_df["Model"],
        rotation=20,
        ha="right"
    )
    ax2.legend()

    # -------------------------
    # Chart 3: Top-K overlap
    # -------------------------

    ax3 = axes[1, 0]

    top_a = set(model_a_df.head(k)["ProductId"])
    top_b = set(model_b_df.head(k)["ProductId"])
    top_c = set(model_c_df.head(k)["ProductId"])

    overlap_labels = [
        "A vs B",
        "A vs C",
        "B vs C"
    ]

    overlap_values = [
        len(top_a & top_b) / k,
        len(top_a & top_c) / k,
        len(top_b & top_c) / k
    ]

    ax3.bar(overlap_labels, overlap_values)
    ax3.set_ylim(0, 1)
    ax3.set_title(f"Top-{k} Overlap Rate")
    ax3.set_ylabel("Overlap rate")

    # -------------------------
    # Chart 4: Top-rank normalized scores
    # -------------------------

    ax4 = axes[1, 1]

    top_n = 10

    a = model_a_df.head(top_n).copy()
    b = model_b_df.head(top_n).copy()
    c = model_c_df.head(top_n).copy()

    a["Rank"] = range(1, len(a) + 1)
    b["Rank"] = range(1, len(b) + 1)
    c["Rank"] = range(1, len(c) + 1)

    a["NormalizedScore"] = safe_minmax(a["Score_A"])
    b["NormalizedScore"] = safe_minmax(b["Score_B"])
    c["NormalizedScore"] = safe_minmax(c["Score_C"])

    ax4.plot(
        a["Rank"],
        a["NormalizedScore"],
        marker="o",
        label="Model A"
    )

    ax4.plot(
        b["Rank"],
        b["NormalizedScore"],
        marker="o",
        label="Model B"
    )

    ax4.plot(
        c["Rank"],
        c["NormalizedScore"],
        marker="o",
        label="Model C"
    )

    ax4.set_title("Top-10 Normalized Score Trend")
    ax4.set_xlabel("Rank position")
    ax4.set_ylabel("Normalized score")
    ax4.set_xticks(range(1, top_n + 1))
    ax4.legend()

    plt.suptitle(
        "Popularity Models Evaluation Dashboard",
        fontsize=16
    )

    plt.tight_layout()

    path = f"{CHART_DIR}/dashboard_popularity_summary.png"
    plt.savefig(path, dpi=150)
    plt.close()

    print("Saved dashboard:", path, flush=True)


def save_charts(
    metrics_df,
    rating_df,
    model_a_df,
    model_b_df,
    model_c_df
):
    """
    Main chart function.
    This version groups charts instead of saving many separate images.
    """

    if SAVE_GROUPED_DASHBOARD:
        save_dashboard_chart(
            metrics_df,
            rating_df,
            model_a_df,
            model_b_df,
            model_c_df,
            PRIMARY_K
        )

    save_grouped_ranking_metrics_chart(
        metrics_df,
        PRIMARY_K
    )

    save_grouped_error_chart(
        rating_df
    )

    save_grouped_top_rank_score_chart(
        model_a_df,
        model_b_df,
        model_c_df,
        top_n=10
    )

    save_grouped_topk_overlap_chart(
        model_a_df,
        model_b_df,
        model_c_df,
        PRIMARY_K
    )

In [73]:
# ============================================================
# STANDARD COMPARISON WITH CF / CB / SVD
# ============================================================

def load_external_model_metrics():
    frames = []
    for path in EXTERNAL_MODEL_METRIC_FILES:
        if os.path.exists(path):
            df = pd.read_csv(path)
            frames.append(df)
            print(f"Loaded external metrics: {path}", flush=True)
        else:
            print(f"External metrics not found, skipped: {path}", flush=True)

    if frames:
        return pd.concat(frames, ignore_index=True)

    return pd.DataFrame()


def create_external_metrics_template():
    template_cols = [
        "Model",
        "K",
        "EvaluatedUsers",
        "Precision@K",
        "Recall@K",
        "F1@K",
        "HitRate@K",
        "MAP@K",
        "MRR@K",
        "NDCG@K",
        "CatalogCoverage@K",
        "RMSE",
        "MAE",
        "RatingEvalRows",
    ]

    examples = pd.DataFrame([
        {
            "Model": "CF",
            "K": PRIMARY_K,
            "EvaluatedUsers": None,
            "Precision@K": None,
            "Recall@K": None,
            "F1@K": None,
            "HitRate@K": None,
            "MAP@K": None,
            "MRR@K": None,
            "NDCG@K": None,
            "CatalogCoverage@K": None,
            "RMSE": None,
            "MAE": None,
            "RatingEvalRows": None,
        },
        {
            "Model": "Content_Based",
            "K": PRIMARY_K,
            "EvaluatedUsers": None,
            "Precision@K": None,
            "Recall@K": None,
            "F1@K": None,
            "HitRate@K": None,
            "MAP@K": None,
            "MRR@K": None,
            "NDCG@K": None,
            "CatalogCoverage@K": None,
            "RMSE": None,
            "MAE": None,
            "RatingEvalRows": None,
        },
        {
            "Model": "SVD",
            "K": PRIMARY_K,
            "EvaluatedUsers": None,
            "Precision@K": None,
            "Recall@K": None,
            "F1@K": None,
            "HitRate@K": None,
            "MAP@K": None,
            "MRR@K": None,
            "NDCG@K": None,
            "CatalogCoverage@K": None,
            "RMSE": None,
            "MAE": None,
            "RatingEvalRows": None,
        },
    ], columns=template_cols)

    path = f"{REPORT_DIR}/external_model_metrics_template.csv"
    examples.to_csv(path, index=False)
    print("Saved external metrics template:", path, flush=True)

In [74]:
# ============================================================
# REPORT
# ============================================================

def write_markdown_report(metrics_df, rating_df, combined_df, rank_compare_df):
    primary_metrics = metrics_df[metrics_df["K"] == PRIMARY_K].copy()
    primary_combined = combined_df[combined_df["K"] == PRIMARY_K].copy() if "K" in combined_df.columns else combined_df

    best_map = primary_metrics.sort_values("MAP@K", ascending=False).head(1)
    best_rmse = rating_df.sort_values("RMSE", ascending=True).head(1)

    lines = []
    lines.append("# Popularity Model Training Report")
    lines.append("")
    lines.append("## Models")
    lines.append("")
    lines.append("- Model A: Popularity Only, `Score = View + 3*Cart + 5*Purchase`.")
    lines.append("- Model B: Popularity + Time Decay, `Score = PopularityScore * DecayFactor`.")
    lines.append("- Model C: Rank-Based Popularity + Bayesian Average, `Score = 0.5*PopularityDecayNorm + 0.5*BayesianNorm`.")
    lines.append("")
    lines.append("## Ranking metrics")
    lines.append("")
    lines.append(primary_metrics.to_markdown(index=False))
    lines.append("")
    lines.append("## Rating prediction metrics")
    lines.append("")
    lines.append(rating_df.to_markdown(index=False))
    lines.append("")

    if len(best_map) > 0:
        row = best_map.iloc[0]
        lines.append(f"Best MAP@{PRIMARY_K}: **{row['Model']}** = {row['MAP@K']:.6f}.")
    if len(best_rmse) > 0:
        row = best_rmse.iloc[0]
        lines.append(f"Best RMSE: **{row['Model']}** = {row['RMSE']:.6f}.")

    lines.append("")
    lines.append("## Files generated")
    lines.append("")
    lines.append("- `model_metrics_popularity.csv`: ranking + rating metrics for Popularity A/B/C.")
    lines.append("- `model_metrics_all_models.csv`: popularity metrics plus optional CF/CB/SVD metrics if provided.")
    lines.append("- `ranking_comparison_popularity.csv`: rank movement across A/B/C.")
    lines.append("- `top_products_all_popularity_models.csv`: top ranked products for each Popularity model.")
    lines.append("- `charts/`: comparison charts for report/presentation.")
    lines.append("")
    lines.append("## Notes for team comparison")
    lines.append("")
    lines.append("CF, Content-Based, and SVD should export metrics using the same columns as `external_model_metrics_template.csv`. Then add their metric file paths to `EXTERNAL_MODEL_METRIC_FILES` and rerun this script to generate a combined comparison report.")

    path = f"{REPORT_DIR}/popularity_training_report.md"
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("Saved markdown report:", path, flush=True)


In [75]:
# ============================================================
# MAIN
# ============================================================

def main():
    # Ensure correct paths are used within this function's scope
    global POPULARITY_TRAIN_FILE, INTERACTION_TRAIN_FILE, INTERACTION_TEST_FILE
    POPULARITY_TRAIN_FILE = "/content/data/popularity_train_dataset.csv"
    INTERACTION_TRAIN_FILE = "/content/data/interaction_train.csv"
    INTERACTION_TEST_FILE = "/content/data/interaction_test.csv"

    ensure_dirs()

    print("===== LOAD DATA =====", flush=True)
    product_df = load_popularity_train_dataset()

    print("\n===== TRAIN MODELS =====", flush=True)
    model_a_df = train_model_a_popularity_only(product_df)
    model_b_df = train_model_b_popularity_time_decay(product_df)
    model_c_df = train_model_c_rank_based_bayesian(product_df)

    print("Model A top 5:", model_a_df.head(5)["ProductId"].tolist(), flush=True)
    print("Model B top 5:", model_b_df.head(5)["ProductId"].tolist(), flush=True)
    print("Model C top 5:", model_c_df.head(5)["ProductId"].tolist(), flush=True)

    print("\n===== LOAD EVALUATION DATA =====", flush=True)
    train_seen_by_user = load_train_seen_items()
    actual_by_user, _ = load_test_actual_items()

    if len(actual_by_user) == 0:
        raise ValueError("No users with relevant items found in interaction_test.csv. Check IsRelevant or Rating threshold.")

    ranked_lists = build_ranked_lists(model_a_df, model_b_df, model_c_df)

    print("\n===== EVALUATE RANKING METRICS =====", flush=True)
    metric_frames = []
    user_metric_frames = []
    sample_frames = []

    for model_name, ranked_products in ranked_lists.items():
        summary_df, user_metrics_df, sample_df = evaluate_ranking_model(
            model_name,
            ranked_products,
            actual_by_user,
            train_seen_by_user,
            TOP_K_LIST,
        )
        metric_frames.append(summary_df)
        if SAVE_USER_METRICS:
            user_metric_frames.append(user_metrics_df)
        if SAVE_RECOMMENDATION_SAMPLE:
            sample_frames.append(sample_df)

    ranking_metrics_df = pd.concat(metric_frames, ignore_index=True)

    print("\n===== EVALUATE RMSE / MAE =====", flush=True)
    prediction_maps = build_prediction_maps(model_a_df, model_b_df, model_c_df)
    rating_metrics_df = evaluate_rating_prediction(prediction_maps)

    metrics_df = ranking_metrics_df.merge(rating_metrics_df, on="Model", how="left")

    print("\n===== BUILD ANALYSIS OUTPUTS =====", flush=True)
    rank_compare_df = build_rank_comparison(model_a_df, model_b_df, model_c_df)

    top_products_df = pd.concat([
        build_top_products(model_a_df, "Popularity_A_Only", "Score_A", "Rank_A"),
        build_top_products(model_b_df, "Popularity_B_TimeDecay", "Score_B", "Rank_B"),
        build_top_products(model_c_df, "Popularity_C_Bayesian", "Score_C", "Rank_C"),
    ], ignore_index=True)

    external_df = load_external_model_metrics()
    if len(external_df) > 0:
        combined_df = pd.concat([metrics_df, external_df], ignore_index=True, sort=False)
    else:
        combined_df = metrics_df.copy()

    create_external_metrics_template()

    print("\n===== SAVE FILES =====", flush=True)
    model_a_df.to_csv(f"{REPORT_DIR}/model_a_popularity_only_result.csv", index=False)
    model_b_df.to_csv(f"{REPORT_DIR}/model_b_popularity_time_decay_result.csv", index=False)
    model_c_df.to_csv(f"{REPORT_DIR}/model_c_bayesian_rank_based_result.csv", index=False)

    metrics_df.to_csv(f"{REPORT_DIR}/model_metrics_popularity.csv", index=False)
    combined_df.to_csv(f"{REPORT_DIR}/model_metrics_all_models.csv", index=False)
    rank_compare_df.to_csv(f"{REPORT_DIR}/ranking_comparison_popularity.csv", index=False)
    top_products_df.to_csv(f"{REPORT_DIR}/top_products_all_popularity_models.csv", index=False)

    if SAVE_USER_METRICS and user_metric_frames:
        user_metrics_all = pd.concat(user_metric_frames, ignore_index=True)
        user_metrics_all.to_csv(f"{REPORT_DIR}/user_metrics_popularity.csv", index=False)

    if SAVE_RECOMMENDATION_SAMPLE and sample_frames:
        samples_all = pd.concat(sample_frames, ignore_index=True)
        samples_all.to_csv(f"{REPORT_DIR}/recommendation_samples_popularity.csv", index=False)

    print("\n===== SAVE CHARTS =====", flush=True)
    save_charts(metrics_df, rating_metrics_df, model_a_df, model_b_df, model_c_df)

    write_markdown_report(metrics_df, rating_metrics_df, combined_df, rank_compare_df)

    print("\n===== DONE =====", flush=True)
    print("Main metrics file:", f"{REPORT_DIR}/model_metrics_popularity.csv", flush=True)
    print("Combined metrics file:", f"{REPORT_DIR}/model_metrics_all_models.csv", flush=True)
    print("Report:", f"{REPORT_DIR}/popularity_training_report.md", flush=True)


if __name__ == "__main__":
    main()

===== LOAD DATA =====
Loaded popularity train products: 1,366,449

===== TRAIN MODELS =====
Model A top 5: ['B01LSUQSB0', 'B0BVGHXZJ1', 'B0BYBCFHTB', 'B0BS71PXPX', 'B0C777X1L4']
Model B top 5: ['B01LSUQSB0', 'B0BVGHXZJ1', 'B0BYBCFHTB', 'B0BS71PXPX', 'B0C777X1L4']
Model C top 5: ['B01LSUQSB0', 'B0BVGHXZJ1', 'B0C777X1L4', 'B0B6G84457', 'B0B3DB5HTC']

===== LOAD EVALUATION DATA =====
train seen chunk 1: processed 100,000 rows
train seen chunk 2: processed 200,000 rows
train seen chunk 3: processed 300,000 rows
train seen chunk 4: processed 400,000 rows
train seen chunk 5: processed 500,000 rows
train seen chunk 6: processed 600,000 rows
train seen chunk 7: processed 700,000 rows
train seen chunk 8: processed 800,000 rows
train seen chunk 9: processed 900,000 rows
train seen chunk 10: processed 1,000,000 rows
train seen chunk 11: processed 1,100,000 rows
train seen chunk 12: processed 1,200,000 rows
train seen chunk 13: processed 1,300,000 rows
train seen chunk 14: processed 1,400,000 rows